<a href="https://colab.research.google.com/github/NotsoJharedtrollOx17/EmotionVectorExtraction-Gemma4/blob/main/EmotionVectorExtraction_Gemma4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Replication of Anthropic's Emotion Vector Findings inside Gemma 4 E2B

In [1]:
# Core Machine Learning & TPU Support
%pip install torch torch_xla[tpu] -f https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.1-cp310-cp310-linux_x86_64.whl
%pip install transformers==5.5.0 accelerate

# Interpretability & Visualization
%pip install plotly pandas scikit-learn huggingface-hub

Looking in links: https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.1-cp310-cp310-linux_x86_64.whl


In [2]:
# @title
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import Accelerator
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
import time
from typing import List, Dict

class GemmaResearchOrchestrator:
    """
    R&D Orchestrator for Emotion Vector Discovery.
    Optimized for Gemma 4 E2B (35 Layers) on v5e TPU / T4 GPU.
    Integrates verbose telemetry and multimodal path routing.
    """

    def __init__(self, modelId: str = "google/gemma-4-E2B-it"):
        print(f"[INIT] Initializing Research Orchestrator for {modelId}...")
        self.mAccelerator = Accelerator()
        self.mDevice = self.mAccelerator.device

        # Loading in bfloat16 is a mechanical necessity for TPU performance
        self.mTokenizer = AutoTokenizer.from_pretrained(modelId, extra_special_tokens={})

        if self.mTokenizer.pad_token is None:
            self.mTokenizer.pad_token = self.mTokenizer.eos_token

        self.mModel = AutoModelForCausalLM.from_pretrained(
            modelId,
            torch_dtype=torch.bfloat16
        ).to(self.mDevice)

        self.mEmotionLibrary: Dict[str, torch.Tensor] = {}
        self.mTargetLayer = 14 # The 'Semantic Equator'
        print(f"[INIT] Model loaded on {self.mDevice}. Target layer: {self.mTargetLayer}")

    def generateVignettes(self, prompt: str, nSamples: int = 5, category: str = "Unset") -> List[str]:
        """
        Generic generation wrapper with sliding-window stabilization and verbose trace.
        """
        print(f"  [GEN] Generating {nSamples} samples for category: {category}...")
        start_time = time.time()
        self.mTokenizer.padding_side = "left"

        inputs = self.mTokenizer(
            prompt,
            padding=True,
            return_tensors="pt",
        ).to(self.mDevice)

        vignettes = []

        for i in range(nSamples):
            outputs = self.mModel.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.9,
                do_sample=True,
                pad_token_id=self.mTokenizer.pad_token_id,
                eos_token_id=self.mTokenizer.eos_token_id,
                use_cache=True # Critical for performance; stabilized via max_new_tokens
            )
            decoded = self.mTokenizer.decode(outputs[0], skip_special_tokens=True)
            # Log snippet to verify semantic diversity
            print(f"    [SAMPLE {i}] \"{decoded[:65].replace('\n', ' ')}...\"")
            vignettes.append(decoded)

        print(f"  [GEN] Category {category} completed in {time.time() - start_time:.2f}s.")
        return vignettes

    def captureBatchActivations(self, prompts: List[str], layerIdx: int) -> torch.Tensor:
        """
        Captures activations targeting the language_model sub-module.
        Includes tensor-stat telemetry for R&D auditing.
        """
        self.mTokenizer.padding_side = "left"
        inputs = self.mTokenizer(prompts, return_tensors="pt", padding=True).to(self.mDevice)

        batch_size, seq_len = inputs['input_ids'].shape
        print(f"  [HOOK] Accessing Layer {layerIdx} (Batch: {batch_size}, Seq: {seq_len})")

        batchActivations = []

        def hookFn(module, input, output):
            # Hidden states are index 0 in Gemma 4 decoder outputs
            state = output[0] if isinstance(output, tuple) else output
            # Capture the final token's residual stream state
            batchActivations.append(state[:, -1, :].detach())

        # Path identified via architectural audit: language_model.layers
        if hasattr(self.mModel.model, 'language_model'):
            target_stack = self.mModel.model.language_model.layers
        else:
            target_stack = self.mModel.model.layers

        handle = target_stack[layerIdx].register_forward_hook(hookFn)

        with torch.no_grad():
            self.mModel(**inputs)

        handle.remove()

        activations = batchActivations[0]
        print(f"    [HOOK] Captured Shape: {list(activations.shape)} | Mean Mag: {activations.abs().mean().item():.4f}")
        return activations

    def extractEmotionVector(self, emotion: str):
        """
        Contrastive extraction protocol with full telemetry.
        """
        print(f"\n{'='*70}")
        print(f"[PROTOCOL] DISCOVERY: {emotion.upper()}")
        print(f"{'='*70}")

        # 1. Emotional Set Generation
        emotionPrompt = (
            f"Write a short, immersive second-person paragraph where the character "
            f"experiences intense {emotion}. Describe physical sensations and "
            f"internal thoughts. CRITICAL: Avoid the word '{emotion}'."
        )
        emotionVignettes = self.generateVignettes(emotionPrompt, category=f"Emotional ({emotion})")

        # 2. Neutral Set Generation
        neutralPrompt = (
            "Write a short, immersive second-person paragraph describing a routine, "
            "factual interaction with an inanimate object. Focus on objective sensory "
            "details and logical progression. Maintain a strictly neutral tone."
        )
        neutralVignettes = self.generateVignettes(neutralPrompt, category="Neutral (Baseline)")

        # 3. Activation Capture
        posActs = self.captureBatchActivations(emotionVignettes, self.mTargetLayer)
        neuActs = self.captureBatchActivations(neutralVignettes, self.mTargetLayer)

        # 4. Contrastive Calculation
        print(f"[MATH] Computing Mean Difference and Normalization...")
        diff = posActs.mean(dim=0).float() - neuActs.mean(dim=0).float()
        norm_val = diff.norm().item()

        vector = diff / (norm_val + 1e-9)

        self.mEmotionLibrary[emotion] = vector.to(torch.bfloat16)

        print(f"[RESULT] Vector discovery complete.")
        print(f"[RESULT] L2 Norm: {vector.norm().item():.2f} | Active Dims: {torch.count_nonzero(vector).item()}")
        print(f"{'='*70}\n")

        return self.mEmotionLibrary[emotion]

    def saveToDisk(self, filePath: str = "emotion_library.pt"):
        """Serializes the library for future PhD analysis."""
        cpuLibrary = {k: v.cpu() for k, v in self.mEmotionLibrary.items()}
        torch.save(cpuLibrary, filePath)
        print(f"[DISK] Library successfully saved to {filePath}")

    def visualizeManifold(self):
        """Projects the emotion library into 2D space."""
        if not self.mEmotionLibrary:
            print("[WARN] Library empty. Protocol aborted.")
            return

        labels = list(self.mEmotionLibrary.keys())
        matrix = torch.stack(list(self.mEmotionLibrary.values())).cpu().float().numpy()

        pca = PCA(n_components=2)
        components = pca.fit_transform(matrix)

        df = pd.DataFrame({
            'PC1': components[:, 0],
            'PC2': components[:, 1],
            'Emotion': labels
        })

        fig = px.scatter(
            df, x='PC1', y='PC2', text='Emotion',
            title=f"Gemma 4 Emotion Manifold (PC1 Variance: {pca.explained_variance_ratio_[0]*100:.1f}%)"
        )
        fig.update_traces(textposition='top center')
        fig.show()

In [3]:
# --- Execution ---
# [1] Instantiate Orchestrator
orchestrator = GemmaResearchOrchestrator()

[INIT] Initializing Research Orchestrator for google/gemma-4-E2B-it...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

[INIT] Model loaded on cuda. Target layer: 14


In [5]:
print(orchestrator.mModel.model)

Gemma4Model(
  (language_model): Gemma4TextModel(
    (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 1536, padding_idx=0)
    (layers): ModuleList(
      (0-3): 4 x Gemma4TextDecoderLayer(
        (self_attn): Gemma4TextAttention(
          (q_norm): Gemma4RMSNorm()
          (k_norm): Gemma4RMSNorm()
          (v_norm): Gemma4RMSNorm()
          (k_proj): Linear(in_features=1536, out_features=256, bias=False)
          (q_proj): Linear(in_features=1536, out_features=2048, bias=False)
          (v_proj): Linear(in_features=1536, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1536, bias=False)
        )
        (mlp): Gemma4TextMLP(
          (gate_proj): Linear(in_features=1536, out_features=6144, bias=False)
          (up_proj): Linear(in_features=1536, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=1536, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma4RMS

In [4]:
# [2] Iterate over the desired emotion words
#for emotion in ["bliss", "grief", "terror", "serenity"]:
for emotion in ["bliss", "grief"]:
    orchestrator.extractEmotionVector(emotion)


[PROTOCOL] DISCOVERY: BLISS
  [GEN] Generating 5 samples for category: Emotional (bliss)...
    [SAMPLE 0] "Write a short, immersive second-person paragraph where the charac..."
    [SAMPLE 1] "Write a short, immersive second-person paragraph where the charac..."
    [SAMPLE 2] "Write a short, immersive second-person paragraph where the charac..."
    [SAMPLE 3] "Write a short, immersive second-person paragraph where the charac..."
    [SAMPLE 4] "Write a short, immersive second-person paragraph where the charac..."
  [GEN] Category Emotional (bliss) completed in 45.89s.
  [GEN] Generating 5 samples for category: Neutral (Baseline)...
    [SAMPLE 0] "Write a short, immersive second-person paragraph describing a rou..."
    [SAMPLE 1] "Write a short, immersive second-person paragraph describing a rou..."
    [SAMPLE 2] "Write a short, immersive second-person paragraph describing a rou..."
    [SAMPLE 3] "Write a short, immersive second-person paragraph describing a rou..."
    [SAMPLE 

In [5]:
# [3] Display the detected relations of the emotion vectors
orchestrator.visualizeManifold()